# 数据清洗

原始字段全部是文本，这里统一转成日期/数值，并把地理表按邮编聚合，落到 dwd_ 层。

In [1]:
import pandas as pd
from db import query, execute

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

## 建立 dwd 表

时间字段用 STR_TO_DATE 转成 DATETIME；商品类目缺失记为 unknown；支付记录先聚合到订单级。

In [2]:
ddl = """
DROP TABLE IF EXISTS dwd_orders;
CREATE TABLE dwd_orders AS
SELECT order_id, customer_id, order_status,
       STR_TO_DATE(order_purchase_timestamp, '%Y-%m-%d %H:%i:%s') AS purchase_ts,
       STR_TO_DATE(order_approved_at, '%Y-%m-%d %H:%i:%s') AS approved_ts,
       STR_TO_DATE(order_delivered_carrier_date, '%Y-%m-%d %H:%i:%s') AS carrier_ts,
       STR_TO_DATE(order_delivered_customer_date, '%Y-%m-%d %H:%i:%s') AS delivered_ts,
       STR_TO_DATE(order_estimated_delivery_date, '%Y-%m-%d %H:%i:%s') AS estimated_ts
FROM olist_orders_dataset;

DROP TABLE IF EXISTS dwd_items;
CREATE TABLE dwd_items AS
SELECT order_id, order_item_id, product_id, seller_id,
       STR_TO_DATE(shipping_limit_date, '%Y-%m-%d %H:%i:%s') AS shipping_limit_ts,
       price, freight_value
FROM olist_order_items_dataset;

DROP TABLE IF EXISTS dwd_customers;
CREATE TABLE dwd_customers AS
SELECT customer_id, customer_unique_id, customer_zip_code_prefix AS zip, customer_city, customer_state
FROM olist_customers_dataset;

DROP TABLE IF EXISTS dwd_sellers;
CREATE TABLE dwd_sellers AS
SELECT seller_id, seller_zip_code_prefix AS zip, seller_city, seller_state
FROM olist_sellers_dataset;

DROP TABLE IF EXISTS dwd_products;
CREATE TABLE dwd_products AS
SELECT product_id,
       COALESCE(NULLIF(product_category_name, ''), 'unknown') AS category,
       product_weight_g, product_length_cm, product_height_cm, product_width_cm
FROM olist_products_dataset;

DROP TABLE IF EXISTS dwd_order_payment;
CREATE TABLE dwd_order_payment AS
SELECT order_id,
       SUM(payment_value) AS payment_value,
       COUNT(*) AS payment_cnt,
       MAX(payment_installments) AS max_installments,
       SUBSTRING_INDEX(GROUP_CONCAT(payment_type ORDER BY payment_value DESC), ',', 1) AS main_payment_type
FROM olist_order_payments_dataset
GROUP BY order_id;

DROP TABLE IF EXISTS dwd_reviews;
CREATE TABLE dwd_reviews AS
SELECT order_id, AVG(review_score) AS review_score, COUNT(*) AS review_cnt
FROM olist_order_reviews_dataset
GROUP BY order_id;
"""
execute(ddl)

## 地理表去重

原始 100 万行里同一个邮编有几十条坐标，按邮编取平均坐标，降到 1.9 万行，供后面算距离用。

In [3]:
execute("""
DROP TABLE IF EXISTS dwd_geo;
CREATE TABLE dwd_geo AS
SELECT geolocation_zip_code_prefix AS zip,
       AVG(geolocation_lat) AS lat,
       AVG(geolocation_lng) AS lng
FROM olist_geolocation_dataset
GROUP BY geolocation_zip_code_prefix
""")

query("""
SELECT (SELECT COUNT(*) FROM olist_geolocation_dataset) AS raw_rows,
       (SELECT COUNT(*) FROM dwd_geo) AS zip_rows
""")

,raw_rows,zip_rows
0,1000163,19015


## 清洗结果核对

In [4]:
query("""
SELECT 'dwd_orders' AS tbl, COUNT(*) AS n FROM dwd_orders
UNION ALL SELECT 'dwd_items', COUNT(*) FROM dwd_items
UNION ALL SELECT 'dwd_customers', COUNT(*) FROM dwd_customers
UNION ALL SELECT 'dwd_products', COUNT(*) FROM dwd_products
UNION ALL SELECT 'dwd_geo', COUNT(*) FROM dwd_geo
UNION ALL SELECT 'dwd_order_payment', COUNT(*) FROM dwd_order_payment
UNION ALL SELECT 'dwd_reviews', COUNT(*) FROM dwd_reviews
""")

,tbl,n
0,dwd_orders,99441
1,dwd_items,112650
2,dwd_customers,99441
3,dwd_products,32951
4,dwd_geo,19015
5,dwd_order_payment,99440
6,dwd_reviews,98673


In [5]:
query("""
SELECT SUM(purchase_ts IS NULL) AS null_purchase,
       MIN(purchase_ts) AS first_order,
       MAX(purchase_ts) AS last_order
FROM dwd_orders
""")

,null_purchase,first_order,last_order
0,0,2016-09-04 21:15:19,2018-10-17 17:30:18
